# Add ImpactMe outcome probs via saved LR models

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.impactme_probs import add_prob_vector
import pandas as pd

In [ ]:
SEG_TYPES = [
    "turns",
    "pt_noshort",
]
MODEL_NAMES = [
    "Llama-3.3-8B-Instruct",
    "Llama-3.3-70B-Instruct",
]
IMPACT_TYPES = ["self", "parent"]
PROB_AGG_TYPES = ["block", "max", "avg"]

Get Probs

In [ ]:
for impact_type in IMPACT_TYPES:
    print(f"{impact_type}...")
    for model_name in MODEL_NAMES:
        print(f"\t{model_name}...")
        for seg_type in SEG_TYPES:
            print(f"\t\t{seg_type}...")
            embed_path = (
                BASE_PATH
                / "data"
                / "w_embeddings"
                / model_name
                / f"{impact_type}.parquet"
            )
            export_path = (
                BASE_PATH
                / "data"
                / "w_probs"
                / model_name
                / seg_type
                / f"{impact_type}.parquet"
            )
            add_prob_vector(
                embed_path=embed_path,
                export_path=export_path,
                impact_type=impact_type,
                model_name=model_name,
                impactme_seg_type=seg_type,
                make_0=True,
            )

## Check output

In [ ]:
def expand_outcomes(df, prob_col, outcome_list):
    new_cols = pd.DataFrame(df[prob_col].tolist(), columns=outcome_list, index=df.index)
    df = df.join(new_cols)
    df = df.drop(columns=prob_col)
    return df

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 500)
from src.config import BASE_PATH


def get_hists(df_w_probs, outcome_list):
    import matplotlib.pyplot as plt
    import numpy as np

    for outcome in outcome_list:
        col = df_w_probs[outcome]
        if max(col) < 0.9:
            print("CLipping...")
            lo, hi = np.percentile(col, [0, 95])
            col = np.clip(col, lo, hi)

        plt.figure(figsize=(10, 5))

        col.hist(bins=10)
        plt.title(outcome)

        plt.tight_layout()
        plt.show()
        plt.close()

In [ ]:
seg_type = "pt_noshort"
model_name = "Llama-3.3-70B-Instruct"
impact_type = "parent"
w_probs_df = pd.read_parquet(
    BASE_PATH / "data" / "w_probs" / model_name / seg_type / f"{impact_type}.parquet"
)

In [ ]:
agg_type = "max"
prob_col = f"prob_vector_{agg_type}"
from src.utils import IMPACTME_OUTCOMES_LR

df = expand_outcomes(w_probs_df, prob_col, IMPACTME_OUTCOMES_LR)
get_hists(df_w_probs=df, outcome_list=IMPACTME_OUTCOMES_LR)

Compare w/ actual text

In [ ]:
df = expand_outcomes(w_probs_df, prob_col, IMPACTME_OUTCOMES_LR)
outcome = "f1"
condition = df[outcome] > 0
for row in (
    df[condition][["open_text_block", outcome]]
    .sort_values(outcome, ascending=True)
    .itertuples()
):
    print(getattr(row, outcome))
    print(row.open_text_block)
    print("*" * 100)

In [ ]:
for idx, row in enumerate(w_probs_df.itertuples()):
    if idx in [0, 1, 2, 3]:
        continue
    print(row.open_text_block)
    print(row.prob_vector_block)
    print(getattr(row, f"impact_score_{impact_type}"))
    break